In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Main table from your screenshot
TABLE_NAME = "workspace.default.sephora_reviews_enrichedsn"

df = spark.table(TABLE_NAME)

display(df.limit(10))
print("Rows:", df.count())
print("Columns:", len(df.columns))
print(df.columns)


In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import matplotlib.patches as mpatches

# sample = your sampled binary user-item matrix
# if sample is a pandas DataFrame:
data = sample.values if hasattr(sample, "values") else sample

# ensure binary 0/1
data = (data > 0).astype(int)

# compute sparsity
total_cells = data.size
nonzero = np.count_nonzero(data)
density = nonzero / total_cells
sparsity = 1 - density

# colors: 0 = light grey, 1 = red
cmap = ListedColormap(['#E6E6E6', '#D62728'])
norm = BoundaryNorm([-0.5, 0.5, 1.5], cmap.N)

plt.figure(figsize=(8, 6))
plt.imshow(data, aspect='auto', cmap=cmap, norm=norm, interpolation='nearest')

plt.title("User–Item Interaction Matrix (Sparsity)", fontsize=14)
plt.xlabel("Items")
plt.ylabel("Users")

# legend
patch_no = mpatches.Patch(color='#E6E6E6', label='No interaction (0)')
patch_yes = mpatches.Patch(color='#D62728', label='Interaction (1)')
plt.legend(handles=[patch_no, patch_yes], loc='upper right')

# annotate sparsity
plt.text(
    0.01, 0.99,
    f"Sparsity: {sparsity*100:.1f}%  |  Density: {density*100:.1f}%",
    transform=plt.gca().transAxes,
    va='top', ha='left'
)

plt.tight_layout()
plt.show()

In [0]:
num_interactions = df.select("author_id", "product_id").distinct().count()

num_users = df.select("author_id").distinct().count()

num_items = df.select("product_id").distinct().count()

sparsity = 1 - (num_interactions / (num_users * num_items))

print(f"Sparsity: {sparsity*100:.2f}%")

In [0]:
from pyspark.sql import functions as F

# Calculate num_interactions
num_interactions = df.select("author_id", "product_id").distinct().count()

# Count unique users
num_users = df.select("author_id").distinct().count()

# Count unique products and reviews
num_products = df.select("product_id").distinct().count()
num_reviews = df.count()

# Count unique brands
num_brands = df.select("brand_name").distinct().count()

# Count text reviews (non-null review_text)
num_text_reviews = df.filter(F.col("review_text").isNotNull()).count()

print(f"Unique users: {num_users:,}")
print(f"Unique interactions: {num_interactions:,}")
print(f"Unique products: {num_products:,}")
print(f"Unique brands: {num_brands:,}")
print(f"Total reviews: {num_reviews:,}")
print(f"Reviews with text: {num_text_reviews:,}")
print(f"Reviews without text: {num_reviews - num_text_reviews:,}")
print(f"Avg reviews per user: {num_reviews / num_users:.2f}")
print(f"Avg reviews per product: {num_reviews / num_products:.2f}")
print(f"Avg products per brand: {num_products / num_brands:.2f}")

In [0]:
plt.figure()

# keep only users with <= 50 reviews
filtered = pdf[pdf["count"] <= 30]

plt.hist(filtered["count"], bins=30)

plt.title("Number of Reviews per User (<=50)")
plt.xlabel("Reviews")
plt.ylabel("Frequency")

plt.show()

In [0]:
from pyspark.sql import functions as F

# Calculate ratings distribution
ratings_dist = (
    df.groupBy("rating")
    .agg(F.count("*").alias("count"))
    .withColumn("percentage", F.round(100 * F.col("count") / df.count(), 2))
    .orderBy("rating")
)

display(ratings_dist)

# Convert to pandas for plotting
pdf_ratings = ratings_dist.toPandas()

# Plot
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.bar(pdf_ratings["rating"], pdf_ratings["count"], color='steelblue')
plt.title("Ratings Distribution")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.xticks(pdf_ratings["rating"])

# Add percentage labels on bars
for idx, row in pdf_ratings.iterrows():
    plt.text(row["rating"], row["count"], f'{row["percentage"]:.1f}%', 
             ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql import functions as F

# Load the table
df = spark.table("workspace.default.sephora_reviews_enrichedSN")

# Select the specific columns you want
display_df = df.select(
    "row_id",
    "author_id",
    "rating",
    "is_recommended",
    "review_text",
    "review_title",
    "product_id",
    "product_name",
    "ingredients",
    "brand_name",
    "submission_time"
)

display(display_df)